# 🏭 CASO 3 — PRODUCCIÓN INDUSTRIAL
## Visualización de Datos con Streamlit
### 📋 TAREA — Desarrollo desde cero

---

## Contexto del problema

**Empresa:** MetalParts Colombia S.A.S.  
**Sector:** Manufactura de piezas metálicas (tornillería y fijaciones)  
**Sede:** Zona Industrial de Itagüí, Antioquia

MetalParts opera 4 líneas de producción con diferentes máquinas y turnos. La gerencia de operaciones solicita un **dashboard de control de producción** para tomar decisiones basadas en datos. Actualmente los reportes se hacen en Excel manualmente y toman 2 días en prepararse.

**El gerente necesita responder estas preguntas:**
1. ¿Cuál es la eficiencia promedio por línea de producción?
2. ¿En qué turno se producen más defectos?
3. ¿Qué máquina genera más tiempos de paro?
4. ¿Cómo ha evolucionado la producción semana a semana?
5. ¿Cuál es la relación entre temperatura y tasa de defectos?

---

## 📌 Tu entregable

Debes construir:
1. **Este notebook** con el análisis exploratorio completo
2. **Un archivo `caso3_produccion_app.py`** con la app Streamlit funcional
3. **Captura de pantalla** de la app corriendo en tu computador

---

## 📊 Dataset disponible

Archivo: `caso3_produccion_dataset.csv`

| Columna | Tipo | Descripción |
|---|---|---|
| id_orden | str | Identificador de orden de producción |
| fecha_produccion | date | Fecha de la orden |
| linea_produccion | str | Línea A, B, C o D |
| producto | str | Nombre del producto fabricado |
| turno | str | Mañana, Tarde o Noche |
| operador | str | Código del operador |
| maquina | str | Código de la máquina |
| unidades_planificadas | int | Meta de producción |
| unidades_producidas | int | Producción real |
| unidades_defectuosas | int | Unidades con defecto |
| tiempo_ciclo_min | float | Tiempo por unidad (minutos) |
| tiempo_paro_min | float | Minutos de paro no planificado |
| causa_paro | str | Motivo del paro |
| temperatura_c | float | Temperatura del proceso |
| consumo_energia_kwh | float | Consumo energético |
| costo_produccion_cop | float | Costo total de la orden |
| eficiencia_pct | float | % de eficiencia (producido/planificado) |
| tasa_defectos_pct | float | % de defectos sobre producidos |
| semana | int | Número de semana del año |

---

## 🎯 Requisitos mínimos

### En el notebook:
- [ ] Cargar y explorar el dataset (shape, tipos, describe)
- [ ] Calcular al menos 4 KPIs del negocio
- [ ] Crear mínimo 5 visualizaciones Plotly respondiendo las preguntas del gerente
- [ ] Cada gráfica debe tener título y etiquetas claras
- [ ] Identificar al menos 1 insight o hallazgo relevante

### En la app Streamlit:
- [ ] Usar `st.set_page_config()` con título y layout wide
- [ ] Mínimo 2 filtros en el sidebar
- [ ] Mínimo 4 KPIs con `st.metric()`
- [ ] Mínimo 4 gráficas Plotly integradas con `st.plotly_chart()`
- [ ] Usar `st.columns()` para el layout (patrón F o Z)
- [ ] Usar `@st.cache_data` para la carga de datos

---

## 🏆 Puntos extra (opcional)
- Agregar un **selector de rango de fechas** con `st.date_input()`
- Mostrar una tabla de **órdenes con defectos > 10%** como alerta
- Usar `st.tabs()` para organizar las secciones del dashboard
- Agregar `st.download_button()` para exportar el dataset filtrado

---

## ⏰ Tiempo estimado: 45–60 minutos

---

# 🚀 ¡Empieza aquí!

In [28]:
# ══════════════════════════════════════════════════════════════
# PASO 1 — Importar librerías
# ══════════════════════════════════════════════════════════════

# Tu código aquí:
import pandas as pd          # Análisis y manipulación de datos
import numpy as np           # Operaciones numéricas
import plotly.express as px  # Gráficas interactivas de alto nivel
import plotly.graph_objects as go  # Gráficas de bajo nivel (más control)
import os                    # Manejo de rutas del sistema de archivos

# Verificar versiones
print('✅ pandas    :', pd.__version__)
print('✅ numpy     :', np.__version__)
print('✅ plotly    : importado correctamente')
print('🎉 ¡Todo listo para comenzar!')

✅ pandas    : 3.0.2
✅ numpy     : 2.4.4
✅ plotly    : importado correctamente
🎉 ¡Todo listo para comenzar!


In [29]:
# ══════════════════════════════════════════════════════════════
# PASO 2 — Cargar y explorar el dataset
# Archivo: 'caso3_produccion_dataset.csv'
# ══════════════════════════════════════════════════════════════

# Tu código aquí:

df = pd.read_csv('caso3_produccion_dataset.csv')
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
display(df.head())

print('📊 Información del dataset:')
df.info()

print('📈 Estadísticas descriptivas:')
df.describe().round(2)


categoricas = ['linea_produccion', 'producto', 'turno', 'operador', 'maquina', 'causa_paro']
print("\n=== VALORES ÚNICOS (categóricas) ===")
for col in categoricas:
    print(f"{col}: {df[col].unique().tolist()}")

Filas: 160 | Columnas: 19


,id_orden,fecha_produccion,linea_produccion,producto,turno,operador,maquina,unidades_planificadas,unidades_producidas,unidades_defectuosas,tiempo_ciclo_min,tiempo_paro_min,causa_paro,temperatura_c,consumo_energia_kwh,costo_produccion_cop,eficiencia_pct,tasa_defectos_pct,semana
0,OP-2000,2024-06-14,Línea D,Remache 6mm,Tarde,OP-106,Torno-02,2235,1443,91,3.76,30.2,Sin causa,23.6,528.88,2147000.0,64.56,6.31,24
1,OP-2001,2024-03-18,Línea A,Tuerca M8,Tarde,OP-109,CNC-02,3024,4317,66,5.33,61.8,Sin causa,25.0,439.95,4480000.0,100.00,1.53,12
2,OP-2002,2024-07-21,Línea B,Tuerca M8,Tarde,OP-109,CNC-01,3293,4278,103,7.70,114.0,Falla eléctrica,38.0,783.20,1584000.0,100.00,2.41,29
3,OP-2003,2024-11-29,Línea C,Varilla Roscada,Noche,OP-109,CNC-02,2839,2150,87,2.52,103.3,Mantenimiento,18.6,243.94,815000.0,75.73,4.05,48
4,OP-2004,2024-01-25,Línea D,Perno M12,Noche,OP-104,CNC-01,3741,1445,62,3.45,39.0,Falla eléctrica,43.8,579.14,272000.0,38.63,4.29,4


📊 Información del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id_orden               160 non-null    str    
 1   fecha_produccion       160 non-null    str    
 2   linea_produccion       160 non-null    str    
 3   producto               160 non-null    str    
 4   turno                  160 non-null    str    
 5   operador               160 non-null    str    
 6   maquina                160 non-null    str    
 7   unidades_planificadas  160 non-null    int64  
 8   unidades_producidas    160 non-null    int64  
 9   unidades_defectuosas   160 non-null    int64  
 10  tiempo_ciclo_min       160 non-null    float64
 11  tiempo_paro_min        160 non-null    float64
 12  causa_paro             160 non-null    str    
 13  temperatura_c          160 non-null    float64
 14  consumo_energia_kwh    160 non-null    flo

In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 3 — Calcular KPIs
# Sugeridos:
#   - Eficiencia promedio general
#   - Tasa de defectos promedio
#   - Total unidades producidas
#   - Costo total de producción
#   - Tiempo de paro total
# ══════════════════════════════════════════════════════════════

# Tu código aquí:

#Eficiencia promedio general
eficiencia_promedio = df['eficiencia_pct'].mean()
print(f"📊 Eficiencia promedio general:    {eficiencia_promedio:.2f}%")

# TOTAL UNIDADES DEFECTUOSAS (Reemplaza a Tasa de Defectos)
total_defectos = df['unidades_defectuosas'].sum()
print(f"⚠️  Total unidades defectuosas:     {total_defectos:,.0f}")

#Total unidades producidas
total_producidas = df['unidades_producidas'].sum()
print(f"🏭 Total unidades producidas:      {total_producidas:,.0f}")

#Costo total de producción
costo_total = df['costo_produccion_cop'].sum()
print(f"💰 Costo total de producción:      ${costo_total:,.0f} COP")

#Tiempo de paro total
tiempo_paro_total = df['tiempo_paro_min'].sum()
horas_paro = tiempo_paro_total / 60
print(f"⏱️  Tiempo de paro total:           {tiempo_paro_total:,.1f} min ({horas_paro:.1f} h)")


📊 Eficiencia promedio general:    76.72%
⚠️  Tasa de defectos promedio:      5.52%
🏭 Total unidades producidas:      414,675
💰 Costo total de producción:      $396,382,000 COP
⏱️  Tiempo de paro total:           9,100.5 min (151.7 h)


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 4 — Análisis por grupos
# Sugeridos:
#   - Eficiencia por línea de producción
#   - Defectos por turno
#   - Tiempo de paro por máquina
#   - Producción semanal
# ══════════════════════════════════════════════════════════════

# Tu código aquí:

#Eficiencia promedio por línea de producción ──────────
efic_linea = (df.groupby('linea_produccion')['eficiencia_pct']
               .mean()
               .round(2)
               .sort_values(ascending=False)
               .reset_index()
               .rename(columns={'eficiencia_pct': 'eficiencia_promedio'}))
print("=== EFICIENCIA POR LÍNEA ===")
display(efic_linea)

# TOTAL DEFECTOS POR TURNO ─────────────────────────── (Modificado)
defectos_turno = (df.groupby('turno')['unidades_defectuosas']
                   .sum()
                   .sort_values(ascending=False)
                   .reset_index())
print("\n=== TOTAL DEFECTOS POR TURNO ===")
display(defectos_turno)

#Tiempo de paro total por máquina ────────────────────
paro_maquina = (df.groupby('maquina')['tiempo_paro_min']
                 .sum()
                 .round(1)
                 .sort_values(ascending=False)
                 .reset_index()
                 .rename(columns={'tiempo_paro_min': 'tiempo_paro_total_min'}))
paro_maquina['tiempo_paro_horas'] = (paro_maquina['tiempo_paro_total_min'] / 60).round(2)
print("\n=== TIEMPO DE PARO POR MÁQUINA ===")
display(paro_maquina)

#Producción semanal (unidades producidas) ────────────
produccion_semanal = (df.groupby('semana')
                        .agg(
                            unidades_producidas=('unidades_producidas', 'sum'),
                            ordenes=('id_orden', 'count'),
                            eficiencia_promedio=('eficiencia_pct', 'mean')
                        )
                        .round(2)
                        .reset_index())
produccion_semanal['eficiencia_promedio'] = produccion_semanal['eficiencia_promedio'].round(2)
print("\n=== PRODUCCIÓN SEMANAL ===")
display(produccion_semanal)


=== EFICIENCIA POR LÍNEA ===


,linea_produccion,eficiencia_promedio
0,Línea C,80.20
1,Línea B,79.23
2,Línea A,78.44
3,Línea D,72.15



=== DEFECTOS POR TURNO ===


,turno,tasa_defectos_promedio
0,Mañana,6.18
1,Tarde,5.61
2,Noche,4.82



=== TIEMPO DE PARO POR MÁQUINA ===


,maquina,tiempo_paro_total_min,tiempo_paro_horas
0,Torno-02,2115.7,35.26
1,Torno-01,1750.3,29.17
2,CNC-01,1601.0,26.68
3,Prensa-02,1367.3,22.79
4,Prensa-01,1133.9,18.90
5,CNC-02,1132.3,18.87



=== PRODUCCIÓN SEMANAL ===


,semana,unidades_producidas,ordenes,eficiencia_promedio
0,1,3072,1,80.38
1,2,894,1,90.95
2,3,5676,2,97.78
3,4,8213,3,63.47
4,5,22409,8,74.80
5,6,14837,7,79.47
6,7,13143,5,64.14
7,8,3112,2,91.04
8,9,7275,3,72.16
9,10,3520,3,47.79


In [41]:
# ══════════════════════════════════════════════════════════════
# PASO 5 — Visualización 1
# Pregunta a responder: ¿Cuál es la eficiencia por línea?
# Tipo sugerido: Barras o Boxplot
# ══════════════════════════════════════════════════════════════

# Tu código aquí:

fig1 = px.bar(
    efic_linea.sort_values('eficiencia_promedio', ascending=True),
    y                      = 'linea_produccion',
    x                      = 'eficiencia_promedio',
    orientation            = 'h',
    title                  = '🏭 Eficiencia Promedio por Línea de Producción (%)',
    labels                 = {'eficiencia_promedio': 'Eficiencia Promedio (%)', 'linea_produccion': ''},
    color                  = 'eficiencia_promedio',
    color_continuous_scale = 'Blues',
    text_auto              = '.2s'
)
fig1.update_layout(height=350, showlegend=False)
fig1.show()

In [40]:
# ══════════════════════════════════════════════════════════════
# PASO 6 — Visualización 2
# Pregunta a responder: ¿En qué turno hay más defectos?
# Tipo sugerido: Barras agrupadas o Violín
# ══════════════════════════════════════════════════════════════

# Tu código aquí:
fig2 = px.violin(
    df,
    x         = 'turno',
    y         = 'tasa_defectos_pct',
    color     = 'turno',
    box       = True,          # muestra boxplot interno
    points    = 'all',         # muestra todos los puntos
    title     = '⚠️ Distribución de Tasa de Defectos por Turno (%)',
    labels    = {'tasa_defectos_pct': 'Tasa de Defectos (%)', 'turno': 'Turno'},
    color_discrete_sequence = px.colors.qualitative.Pastel
)
fig2.update_layout(height=450, showlegend=False)
fig2.show()

In [45]:
# ══════════════════════════════════════════════════════════════
# PASO 7 — Visualización 3
# Pregunta a responder: ¿Qué máquina genera más paros?
# Tipo sugerido: Barras horizontales
# ══════════════════════════════════════════════════════════════

# Tu código aquí:

fig3 = px.bar(
    paro_maquina.sort_values('tiempo_paro_total_min', ascending=True),
    y                      = 'maquina',
    x                      = 'tiempo_paro_total_min',
    orientation            = 'h',
    title                  = '⏱️ Tiempo de Paro Total por Máquina (min)',
    labels                 = {'tiempo_paro_total_min': 'Tiempo de Paro (min)', 'maquina': ''},
    color                  = 'tiempo_paro_total_min',
    color_continuous_scale = 'Reds',
    text_auto              = '.2s'
)
fig3.update_layout(height=350, showlegend=False)
fig3.show()

In [49]:
# ══════════════════════════════════════════════════════════════
# PASO 8 — Visualización 4
# Pregunta a responder: ¿Cómo evoluciona la producción semanal?
# Tipo sugerido: Línea de tiempo
# ══════════════════════════════════════════════════════════════

# Tu código aquí:

fig4 = px.line(
    produccion_semanal,
    x       = 'semana',
    y       = 'unidades_producidas',
    title   = '📈 Evolución de la Producción Semanal',
    labels  = {'semana': 'Semana del Año', 'unidades_producidas': 'Total de Unidades Producidas'},
    markers = True # Agrega puntos en cada semana para mayor claridad
)
# 3. Estilizamos la línea y los marcadores
fig4.update_traces(
    line_color='#2563eb', 
    line_width=3, 
    marker=dict(size=8, color='#1e40af')
)
# 4. Ajustes visuales de formato
fig4.update_layout(
    height=400,
    xaxis=dict(
        tickmode='linear', 
        dtick=4  # Muestra una etiqueta en el eje X cada 4 semanas para no saturarlo
    ),
    yaxis=dict(gridcolor='#e2e8f0'),
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
)
fig4.show()

In [50]:
# ══════════════════════════════════════════════════════════════
# PASO 9 — Visualización 5
# Pregunta a responder: ¿Temperatura vs tasa de defectos?
# Tipo sugerido: Scatter plot con trendline
# ══════════════════════════════════════════════════════════════

# Tu código aquí:

fig5 = px.scatter(
    df,
    x         = 'temperatura_c',
    y         = 'tasa_defectos_pct',
    color     = 'linea_produccion', # Agregamos color por línea para ver si alguna se comporta distinto
    trendline = 'ols',              # 'ols' genera la línea de tendencia (Ordinary Least Squares)
    title     = '🌡️ Relación entre Temperatura y Tasa de Defectos',
    labels    = {
        'temperatura_c': 'Temperatura (°C)', 
        'tasa_defectos_pct': 'Tasa de Defectos (%)',
        'linea_produccion': 'Línea'
    },
    hover_data = ['id_orden', 'producto', 'turno'] # Información extra al pasar el mouse por un punto
)
fig5.update_layout(
    height=450,
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    xaxis=dict(gridcolor='#e2e8f0'),
    yaxis=dict(gridcolor='#e2e8f0')
)
fig5.show()

## 💡 Espacio para Insights

Escribe aquí tus hallazgos principales (en texto, no código):

1. **Insight 1:** *(Describe qué encontraste en los datos)*
2. **Insight 2:** 
3. **Recomendación para el gerente:** 

---

In [37]:
# ══════════════════════════════════════════════════════════════
# PASO 10 — App Streamlit
# Crea el archivo caso3_produccion_app.py en la misma carpeta
# con toda la app completa según los requisitos de la tarea
# ══════════════════════════════════════════════════════════════

# Puedes usar esta celda para escribir el código de la app
# y luego copiarlo al archivo .py

app_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px

# Tu app aquí:

'''

# Guardar la app en un archivo
with open('caso3_produccion_app.py', 'w') as f:
    f.write(app_code)

print('✅ Archivo caso3_produccion_app.py creado')
print('Ejecuta: streamlit run caso3_produccion_app.py')

✅ Archivo caso3_produccion_app.py creado
Ejecuta: streamlit run caso3_produccion_app.py
